# Task 3 - Agentic Workflows: Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GimsaraK/CDAZZDEV-MLE-Gimsara_Elgiriyage/blob/main/task3_agentic/task3_multi_agent.ipynb)

CDAZZDEV Senior Machine Learning Engineer Technical Assessment - Gimsara Elgiriyage

| Part | What this notebook shows |
|---|---|
| 3A | One LangGraph research agent with five tools, a visible observe-and-replan trace, a validated three-section report, and recovery from injected tool failures |
| 3B | Two-agent pipeline (Data Analyst and Research Writer) |
| 3C | Short-term memory, persistent cache, and `agent_trace.jsonl` |

Agent LLM: Groq `openai/gpt-oss-120b`, then Groq `openai/gpt-oss-20b` (its own daily quota), then free OpenRouter models. The trace shows which model answered each turn. Ticker: **AAPL**. Not investment advice.

## 0. Setup

Install dependencies and load API keys from Colab Secrets or a local `.env` file. No credentials are hardcoded.

In [1]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Colab clones the public repo and installs Task 3 requirements. Locally we only find the repo root.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/GimsaraK/CDAZZDEV-MLE-Gimsara_Elgiriyage.git"
REPO_NAME = "CDAZZDEV-MLE-Gimsara_Elgiriyage"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)
    else:
        # A reused session keeps its old clone. Pull so a fix pushed to main is picked up.
        subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_root / "task3_agentic" / "requirements.txt")],
        check=True,
    )
else:
    here = Path.cwd().resolve()
    repo_root = here if (here / "task3_agentic").exists() else here.parent
# Task 3 imports its own package and the Task 1 modules it reuses, both from the repo root.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Running in Colab:", IN_COLAB)
# The short commit hash shows exactly which version of the code produced these outputs.
commit = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository commit:", commit or "unknown")
print("Repository root :", repo_root)

Running in Colab: True
Repository commit: 0cb38dc
Repository root : /content/CDAZZDEV-MLE-Gimsara_Elgiriyage


In [2]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Keys come from Colab Secrets or the repo-root .env. Only True/False is printed, never a key.
from importlib.metadata import version

from task3_agentic.src.llm import load_api_keys
from task3_agentic.src.logging_utils import configure_logging

configure_logging()
print("API keys present:", load_api_keys())
for package in ("langgraph", "langchain-core", "langchain-groq", "langchain-openai", "ddgs", "yfinance", "pydantic"):
    print(f"{package:>16}: {version(package)}")

API keys present: {'GROQ_API_KEY': True, 'OPENROUTER_API_KEY': True}
       langgraph: 1.2.14
  langchain-core: 1.6.8
  langchain-groq: 1.1.3
langchain-openai: 1.7.0
            ddgs: 9.16.0
        yfinance: 1.7.0
        pydantic: 2.13.5


## 1. Configuration and Trace Logger

Every tool call is written to `logs/agent_trace.jsonl` with its inputs, output (truncated to 200 characters), and wall-clock duration.

Every limit the agent runs under is a named constant in `src/config.py`. The trace logger appends one JSON line per tool call. A full top-to-bottom run of this notebook starts a fresh `logs/agent_trace.jsonl`, so the committed file holds exactly the runs shown below.

In [3]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

import pandas as pd

from task3_agentic.src import config
from task3_agentic.src.tracing import TraceLogger

settings = {
    "Ticker": config.TICKER,
    "Agent LLM": config.GROQ_MODEL,
    # GROQ_MODELS[0] is the primary model; everything after it, then the OpenRouter models, is the fallback order.
    "Fallback LLMs": ", ".join(config.GROQ_MODELS[1:] + config.OPENROUTER_FALLBACK_MODELS),
    "Reasoning effort (agent / report)": f"{config.AGENT_REASONING_EFFORT} / {config.REPORT_REASONING_EFFORT}",
    "Max tool rounds": config.MAX_TOOL_ROUNDS,
    "Extra rounds per report revision": config.REVISION_TOOL_ROUNDS,
    "Max report revisions": config.MAX_REPORT_REVISIONS,
    "Tool result cap sent to the LLM (chars)": config.TOOL_MESSAGE_MAX_CHARS,
    "Trace output truncation (chars)": config.TRACE_OUTPUT_MAX_CHARS,
    "Hedge horizon (calendar days)": config.HEDGE_HORIZON_CALENDAR_DAYS,
    "Volatility method": "std(daily log returns, ddof=1) x sqrt(252)",
    "Hedge vol check tolerance (pct points)": config.VOL_CHECK_TOLERANCE_PCT_POINTS,
}
display(pd.DataFrame(settings.items(), columns=["Setting", "Value"]))

# Fresh trace file for this notebook run: writing "" empties it, so the committed agent_trace.jsonl holds
# exactly the tool calls of this execution. Every later SessionContext shares this one logger.
config.TRACE_FILE.parent.mkdir(parents=True, exist_ok=True)
config.TRACE_FILE.write_text("", encoding="utf-8")
trace = TraceLogger(config.TRACE_FILE)
print("Trace file:", config.TRACE_FILE.relative_to(config.REPO_ROOT))

,Setting,Value
0,Ticker,AAPL
1,Agent LLM,openai/gpt-oss-120b
2,Fallback LLMs,"openai/gpt-oss-20b, nvidia/nemotron-3-super-12..."
3,Reasoning effort (agent / report),low / medium
4,Max tool rounds,8
5,Extra rounds per report revision,3
6,Max report revisions,2
7,Tool result cap sent to the LLM (chars),2500
8,Trace output truncation (chars),200
9,Hedge horizon (calendar days),90


Trace file: task3_agentic/logs/agent_trace.jsonl


In [4]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

# 3C: cache and memory events go to their own log, so agent_trace.jsonl stays one line per tool call.
from task3_agentic.src.tracing import EventLogger, trace_line_count

config.EVENTS_FILE.write_text("", encoding="utf-8")
events = EventLogger(config.EVENTS_FILE)
print("Events file:", config.EVENTS_FILE.relative_to(config.REPO_ROOT))

Events file: task3_agentic/logs/session_events.jsonl


# Part 3A - Tool-Using Research Agent

## 2. Tool Implementations

`get_price_data`, `get_news`, `calculate_volatility`, `llm_sentiment`, `web_search`.

Each tool returns a `ToolResult` envelope: `status` (`ok` / `empty` / `error`), a typed payload in `data`, and on failure an `error` plus a `hint` that names an alternative. Tools never raise. Any exception becomes an `error` envelope inside `run_traced`, which also times the call and writes the trace line.

| Tool | Payload type | Built on | Fallbacks inside the tool |
|---|---|---|---|
| `get_price_data(ticker, period)` | `PriceData` | Task 1 `load_market_data`, `add_indicators`, `build_summary` + yfinance `.info` | live -> Task 1 snapshot (read-only); bad period -> error with the valid list |
| `get_news(ticker, n)` | `NewsList` | Task 1 `get_headlines` | yfinance -> Google News RSS -> snapshot |
| `calculate_volatility(ticker, window)` | `VolatilityResult` | `src/volatility.py` | reuses prices already in the session; bad window -> error with the valid range |
| `llm_sentiment(headlines)` | `SentimentScore` | Task 1 `score_headlines` (Pydantic-validated, repair + provider fallback) | empty list -> hint to call `get_news` |
| `web_search(query)` | `SearchResults` | `ddgs` (the renamed `duckduckgo-search` package) | text -> news backend -> simplified query; retries on rate limits |

The cell below calls each tool directly, outside the agent, and prints the returned types.

In [5]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.display import summarize_result
from task3_agentic.src.session import SessionContext
from task3_agentic.src.tools import ResearchTools

# Call each tool directly (no LLM involved) to show it works on its own. agent_name="tool_check" labels
# these calls in the trace, so they are not mistaken for agent decisions.
tool_ctx = SessionContext(config.TICKER, trace=trace, agent_name="tool_check")
toolkit = ResearchTools(tool_ctx)

price = toolkit.get_price_data(config.TICKER, "1y")
news = toolkit.get_news(config.TICKER, 10)
volatility = toolkit.calculate_volatility(config.TICKER, 20)
# Only the headline titles are passed. If get_news failed, an empty list makes llm_sentiment return an
# "empty" envelope instead of scoring nothing.
sentiment = toolkit.llm_sentiment([h.title for h in news.data.headlines] if news.ok else [])
search = toolkit.web_search("Apple AAPL analyst price target outlook")

rows = []
for result in (price, news, volatility, sentiment, search):
    rows.append({
        "tool": result.tool,
        # Shows the typed return value, e.g. ToolResult[PriceData]: the envelope plus the data model inside it.
        "returns": f"{type(result).__name__}[{type(result.data).__name__}]",
        "status": result.status,
        "source": result.source,
        "summary": summarize_result(result),
    })
with pd.option_context("display.max_colwidth", 160):
    display(pd.DataFrame(rows))

,tool,returns,status,source,summary
0,get_price_data,ToolResult[PriceData],ok,live,"Apple Inc.: price 340.42, 1y return 32.4%, RSI 61.23, momentum Strong Bullish, fwd P/E 35.506, margin 27.62% [source=live]"
1,get_news,ToolResult[NewsList],ok,live,"10 headlines from ['google_rss']; newest: 'Why Are Nasdaq Futures Surging Premarket? SPCX, ASTS, AAPL, TMUS, VZ, NVDA, MU, LITE Stocks In Focus'"
2,calculate_volatility,ToolResult[VolatilityResult],ok,session,"20d vol 16.8% (low, 17.5 pctile 1y); 90d 1-sigma move +/-8.33% -> band 313.2-370.0"
3,llm_sentiment,ToolResult[SentimentScore],ok,groq/openai/gpt-oss-120b,score 0.4134 (Positive) from 10 headlines: 5 positive / 1 negative / 4 neutral
4,web_search,ToolResult[SearchResults],ok,ddgs,"6 hits via text for 'Apple AAPL analyst price target outlook'; top: 'Apple (AAPL) Stock Forecast, Price Targets and Analysts ...Apple (AAPL) Stock Forecast ..."


In [6]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# What the agent actually receives: compact JSON, capped at TOOL_MESSAGE_MAX_CHARS characters.
import json

print(f"calculate_volatility -> {len(volatility.to_llm_json())} chars")
# Parse the compact one-line JSON and print it indented, only to make it readable here.
print(json.dumps(json.loads(volatility.to_llm_json()), indent=2))
print(f"\nget_price_data fundamentals block:")
print(json.dumps(price.data.fundamentals.model_dump(exclude_none=True), indent=2) if price.ok else price.error)

calculate_volatility -> 527 chars
{
  "status": "ok",
  "tool": "calculate_volatility",
  "data": {
    "ticker": "AAPL",
    "window": 20,
    "as_of_date": "2026-10-08",
    "method": "std(daily log returns, ddof=1) x sqrt(252)",
    "annualised_vol_pct": 16.8,
    "daily_vol_pct": 1.06,
    "comparison_pct": {
      "20d": 16.8,
      "60d": 26.36,
      "252d": 24.74
    },
    "percentile_1y": 17.5,
    "regime": "low",
    "spot": 340.42,
    "horizon_calendar_days": 90,
    "horizon_trading_days": 62,
    "expected_move_pct": 8.33,
    "band_1sigma_low": 313.2,
    "band_1sigma_high": 370.0,
    "band_2sigma_low": 288.16,
    "max_drawdown_1y_pct": -13.8
  },
  "warnings": [],
  "source": "session"
}

get_price_data fundamentals block:
{
  "market_cap": 4968150990848.0,
  "trailing_pe": 39.039,
  "forward_pe": 35.506,
  "profit_margin_pct": 27.62,
  "revenue_growth_pct": 16.4,
  "earnings_growth_pct": 28.7,
  "return_on_equity_pct": 148.75,
  "debt_to_equity_pct": 78.445,
  "cur

In [7]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Bad inputs come back as error envelopes with a hint, never as exceptions.
for result in (
    toolkit.get_price_data(config.TICKER, "7y"),
    toolkit.calculate_volatility(config.TICKER, 1000),
    toolkit.llm_sentiment([]),
    toolkit.get_price_data("NOT A TICKER!", "1y"),
):
    print(f"{result.tool:<22} status={result.status:<6} error={result.error}")
    print(f"{'':<22} hint={result.hint}\n")

10:53:47 | WARNING | task3.tracing | Tool get_price_data failed: ValueError: 'NOT A TICKER!' is not a valid ticker symbol
get_price_data         status=error  error=Unsupported period '7y'
                       hint=Use one of: 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max

calculate_volatility   status=error  error=window must be between 5 and 252 trading days, got 1000
                       hint=Use an integer window between 5 and 252, e.g. 20 or 60.

llm_sentiment          status=empty  error=No headlines to score
                       hint=Call get_news first and pass its headline titles as a list of strings.

get_price_data         status=error  error=ValueError: 'NOT A TICKER!' is not a valid ticker symbol
                       hint=Retry once (network errors are often transient). If it fails again, check the ticker symbol and use get_news or web_search for qualitative evidence while prices are unavailable.



## 3. Research Agent

The agent is a LangGraph `StateGraph` built by hand (`src/agent.py`):

```
START -> agent --(tool calls)--> tools -> agent             the LLM picks every call
           |
           +--(no tool calls / budget spent / LLM down)--> write_report -> check_report
                                                               |-- passes ---------------> END
                                                               |-- gaps, revisions left -> agent  (REPLAN)
                                                               +-- gaps, none left ------> END (warnings kept)
```

- **agent**: the LLM with all five tools bound (`bind_tools`). The system prompt states the goal and what each tool returns. It does **not** prescribe an order. Routing only follows the model's own tool calls.
- **tools**: LangGraph `ToolNode`. Arguments that fail the schema, or an unknown tool name, come back to the model as an error message.
- **write_report**: structured output into the Pydantic `ResearchReport` (exactly three risks, each with evidence), written from a compact digest of this session's tool results.
- **check_report**: deterministic checks, no LLM. Each risk must cite a tool that succeeded. The hedge's volatility numbers must match a `calculate_volatility` result. Any tool that only failed must be named in `data_gaps`. A failure sends the agent back with the specific gaps, which is a replan cycle.
- Limits: 8 tool rounds (plus 3 per revision), 2 revisions, LangGraph recursion limit 60. If both LLM providers are down, the run ends with `status="failed"` instead of an exception.

In [8]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src import prompts
from task3_agentic.src.agent import build_research_graph, default_runnables
from task3_agentic.src.tools import build_tools

# This graph is built only to print its structure. TraceLogger(None) keeps records in memory, so nothing
# from this preview is written to agent_trace.jsonl, and no LLM is called.
preview_ctx = SessionContext(config.TICKER, trace=TraceLogger(None))
preview_tools = build_tools(preview_ctx)
models = default_runnables(preview_tools)
graph = build_research_graph(preview_ctx, models["agent"], models["report"], preview_tools)
# draw_mermaid() prints the compiled graph (nodes and edges) as a Mermaid diagram definition.
print(graph.get_graph().draw_mermaid())
print("Tools bound to the agent:", [t.name for t in preview_tools])
print("\n--- Agent system prompt ---")
print(prompts.AGENT_SYSTEM.format(max_rounds=config.MAX_TOOL_ROUNDS, today="<run date>", ticker=config.TICKER))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	write_report(write_report)
	check_report(check_report)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	agent -.-> write_report;
	check_report -.-> __end__;
	check_report -.-> agent;
	tools --> agent;
	write_report --> check_report;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

Tools bound to the agent: ['get_price_data', 'get_news', 'calculate_volatility', 'llm_sentiment', 'web_search']

--- Agent system prompt ---
You are an autonomous equity research agent. Answer the user's research question with evidence you gather yourself using your tools.

Your tools:
- get_price_data(ticker, period): prices, technical indicators, momentum, 52-week range, fundamentals.
- get_news(ticker, n): recent headlines.
- calculate_volatility(ticker, window): annualised histo

**Reading the graph.** The Mermaid text above is the compiled 3A graph. Solid arrows always run; dotted arrows are decisions made at run time. After each turn the agent either calls tools (`tools`, which always returns to `agent`) or, once it has enough evidence, goes to `write_report`. `check_report` then ends the run or sends the agent back for more evidence (the replan loop). The direct `agent` to `__end__` edge is used only by the section 10 follow-up question.

## 4. Agent Run: Observe and Replan Trace

The query from the specification, on AAPL. The trace prints live:

- **DECIDE**: the model's note for its next move. gpt-oss returns tool calls with empty text, so its reasoning summary is shown instead, marked `(reasoning)`.
- **CALL**: the tool and the arguments the model chose.
- **OBSERVE**: the tool's status and key numbers.

Every DECIDE after a tool result is numbered as a cycle and names the observation it followed. **REPLAN** marks a report check that sent the agent back.

In [9]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.agent import run_research
from task3_agentic.src.display import TracePrinter

ctx = SessionContext(config.TICKER, trace=trace)
# The agent runs on its own from here. TracePrinter is called with every new message as it arrives,
# which prints the live DECIDE / CALL / OBSERVE trace below.
run = run_research(ctx, on_message=TracePrinter())
print("\nStatus:", run.final.status, "| revisions:", run.final.revisions, "| duration:", run.duration_s, "s")
print("Agent turns by model:", run.final.agent_models, "| report written by:", run.final.report_model)
print("Graph events:", run.events or "none")

----------------------------------------------------------------------------------------------------
QUERY    Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
----------------------------------------------------------------------------------------------------
DECIDE   [step 1 | openai/gpt-oss-120b]
         (reasoning) We need to gather data.Observed: need price data, news, volatility, sentiment, web search. -> Next: call get_price_data for AAPL 1y.
CALL     get_price_data({"period": "1y", "ticker": "AAPL"})
OBSERVE  get_price_data -> Apple Inc.: price 340.42, 1y return 32.4%, RSI 61.23, momentum Strong Bullish, fwd P/E 35.506, margin 27.62% [source=live]
DECIDE   [step 2 | openai/gpt-oss-120b] cycle 1: after observing get_price_data
         (reasoning) We have price data. Need news, sentiment, volatility, web search.
CALL     get_news({"n": 10, "ticker":

In [10]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from collections import Counter

from task3_agentic.src.display import tool_call_table

# The order below was chosen by the model at run time; nothing in the code fixes it.
display(tool_call_table(ctx))
print("Tool order chosen by the agent:", " -> ".join(run.tool_sequence))
print("Distinct tools used:", len(set(run.tool_sequence)), "of 5 |", dict(Counter(run.tool_sequence)))

,#,agent,tool,args,status,source,duration_ms
0,1,research_agent,get_price_data,"{""ticker"": ""AAPL"", ""period"": ""1y""}",ok,live,635
1,2,research_agent,get_news,"{""ticker"": ""AAPL"", ""n"": 10}",ok,live,594
2,3,research_agent,llm_sentiment,"{""headlines"": [""Why Are Nasdaq Futures Surging...",ok,groq/openai/gpt-oss-120b,31154
3,4,research_agent,calculate_volatility,"{""ticker"": ""AAPL"", ""window"": 60}",ok,session,5
4,5,research_agent,web_search,"{""query"": ""Apple analyst downgrade risk 2026 Q...",ok,ddgs,1822


Tool order chosen by the agent: get_price_data -> get_news -> llm_sentiment -> calculate_volatility -> web_search
Distinct tools used: 5 of 5 | {'get_price_data': 1, 'get_news': 1, 'llm_sentiment': 1, 'calculate_volatility': 1, 'web_search': 1}


In [11]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from IPython.display import Markdown

from task3_agentic.src.report import render_markdown, save_report

company = ctx.company_names.get(config.TICKER)
md_path, json_path = save_report(run.final, company)
print("Saved:", md_path.relative_to(config.REPO_ROOT), "and", json_path.name)
display(Markdown(render_markdown(run.final, company)))

Saved: task3_agentic/outputs/AAPL_2026-10-09_research_report.md and AAPL_2026-10-09_research_report.json


# Apple Inc. (AAPL) - Agent Research Report

*Generated 2026-10-09T10:56:05+00:00 | session `3e5a9005bdcd` | status: **validated** | report revisions: 0 | tools used: calculate_volatility, get_news, get_price_data, llm_sentiment, web_search*

*Agent turns by model: openai/gpt-oss-120b x5, openai/gpt-oss-20b x1 | report written by: openai/gpt-oss-120b*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## 1. Financial Health Summary

Apple is trading at $340.42, just 1.4% below its 52‑week high of $345.34, and sits above both the 50‑day SMA ($322.29) and the 200‑day SMA ($290.07), confirming a strong uptrend. Momentum indicators show an RSI of 61.2 (neutral‑to‑overbought) and a slightly negative MACD histogram, hinting at a short‑term pull‑back risk. Valuation remains premium with a trailing P/E of 39.0× and a forward P/E of 35.5×, while profitability is robust – profit margin is 27.6% and earnings growth 28.7% YoY. The balance sheet is solid with a debt‑to‑equity ratio of 78.4%, a current ratio of 1.00 and free cash flow of $107.7 bn, providing ample liquidity despite moderate leverage.

**Market sentiment.** The LLM‑scored news sentiment is +0.34, indicating a positive tilt. Headlines such as “Apple gains share even as PC market slumps in Q3, IDC says” and “Apple (AAPL) Has a New Catalyst That Could Drive Its Next Upgrade Cycle” were flagged positive, while “Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates” was flagged negative, tempering optimism.

| Metric | Value | Source tool |
|---|---|---|
| Current price | $340.42 | `get_price_data` |
| Trailing P/E | 39.0× | `get_price_data` |
| Profit margin | 27.6% | `get_price_data` |
| Revenue growth YoY | 16.4% | `get_price_data` |
| Free cash flow | $107.7 bn | `get_price_data` |

## 2. Top Three Risks (next 90 days)

### Risk 1: Valuation compression

Apple’s stock trades at a high multiple (trailing P/E 39.0×) and analysts note it may be overvalued relative to cash‑flow estimates, which could trigger a price correction.

Evidence:
- `get_price_data`: trailing_pe = 39.039
- `get_news`: Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates

### Risk 2: Volatility‑driven price swing

Elevated annualised volatility of 26.36% implies a 90‑day 1‑sigma move of about 13.1%, meaning the share could swing between roughly $298.7 and $387.9.

Evidence:
- `calculate_volatility`: annualised_vol_pct = 26.36
- `calculate_volatility`: expected_move_pct = 13.08
- `calculate_volatility`: band_1sigma_low = 298.69

### Risk 3: Earnings guidance disappointment

Recent commentary highlights a “sell the news” reaction after a modest earnings beat, suggesting the stock could fall if guidance falls short of expectations.

Evidence:
- `web_search`: stock fell 7.35% in response, a classic "sell the news" pattern suggesting investors demand stronger forward guidance ([source](https://www.tradingpedia.com/2026/10/02/apple-tests-key-technical-level-ahead-of-q4-earnings/))

## 3. Hedge Strategy Recommendation

**Strategy:** protective put  
**Instrument:** AAPL 90‑day put, strike near 1‑sigma low $298.69  
**Volatility basis:** annualised 26.36%, 90-day 1-sigma move 13.08%

**Rationale.** The put caps downside from valuation compression, volatility swings and potential earnings‑guidance disappointment while preserving upside potential.

**Sizing.** strike $298.69, hedge 1 put contract per 100 shares owned

**Trade-offs.** Premium paid reduces net return; upside beyond $298.69 is forfeited if the stock rallies.

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


## 5. Tool Failure and Fallback Demonstration

The same query again, with two failures injected through `FaultConfig`:

- `web_search` **always** fails with a simulated DuckDuckGo rate limit.
- `get_price_data` fails on its **first** call with a simulated Yahoo timeout.

Faults are raised inside `run_traced`, the same path a real exception takes. So the agent sees the same `error` envelope and hint it would get in production. Nothing tells the agent which tools are broken. It has to read the observation and pick an alternative.

In [12]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.session import FaultConfig

# Planned failures: every web_search call fails, and only the first get_price_data call fails (so a
# retry succeeds). The failures happen inside the tool wrapper, so the agent sees normal error results.
faults = FaultConfig({"web_search": "always", "get_price_data": "first"})
fault_ctx = SessionContext(config.TICKER, trace=trace, faults=faults)
fault_run = run_research(fault_ctx, on_message=TracePrinter())
print("\nStatus:", fault_run.final.status, "| revisions:", fault_run.final.revisions, "| duration:", fault_run.duration_s, "s")
print("Agent turns by model:", fault_run.final.agent_models, "| report written by:", fault_run.final.report_model)
print("Graph events:", fault_run.events or "none")

----------------------------------------------------------------------------------------------------
QUERY    Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
----------------------------------------------------------------------------------------------------
DECIDE   [step 1 | openai/gpt-oss-120b]
         (reasoning) We need to call tools.Observed: need data -> Next: call get_price_data for AAPL 1y.
CALL     get_price_data({"period": "1y", "ticker": "AAPL"})
10:56:15 | WARNING | task3.tracing | Tool get_price_data failed: InjectedFault: Simulated Yahoo Finance failure: read timeout after 30s
OBSERVE  get_price_data -> ERROR: InjectedFault: Simulated Yahoo Finance failure: read timeout after 30s | hint: Retry once (network errors are often transient). If it fails again, check the ticker symbol and use get_news or web_search for qualitative evidence while 

In [13]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.display import trace_table

display(tool_call_table(fault_ctx))
# The same calls as they were written to agent_trace.jsonl (output truncated to 200 characters).
fault_trace = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=fault_ctx.session_id)
# Keep only the calls that did not return "ok" (the injected failures).
failed_calls = fault_trace[fault_trace["status"] != "ok"] if not fault_trace.empty else fault_trace
if failed_calls.empty:
    print("No failed tool calls were logged for this session.")
else:
    with pd.option_context("display.max_colwidth", 90):
        display(failed_calls[["tool", "args", "status", "duration_ms", "output"]])

,#,agent,tool,args,status,source,duration_ms
0,1,research_agent,get_price_data,"{""ticker"": ""AAPL"", ""period"": ""1y""}",error,None,3
1,2,research_agent,get_price_data,"{""ticker"": ""AAPL"", ""period"": ""1y""}",ok,live,462
2,3,research_agent,calculate_volatility,"{""ticker"": ""AAPL"", ""window"": 60}",ok,session,4
3,4,research_agent,get_news,"{""ticker"": ""AAPL"", ""n"": 10}",ok,live,1041
4,5,research_agent,llm_sentiment,"{""headlines"": [""Why Are Nasdaq Futures Surging...",ok,groq/openai/gpt-oss-120b,17782


,tool,args,status,duration_ms,output
0,get_price_data,"{'ticker': 'AAPL', 'period': '1y'}",error,2.6,"{""status"":""error"",""tool"":""get_price_data"",""error"":""InjectedFault: Simulated Yahoo Fina..."


In [14]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

company = fault_ctx.company_names.get(config.TICKER)
fault_md, _ = save_report(fault_run.final, company, suffix="_fault_demo")
print("Saved:", fault_md.relative_to(config.REPO_ROOT))
display(Markdown(render_markdown(fault_run.final, company)))

Saved: task3_agentic/outputs/AAPL_2026-10-09_research_report_fault_demo.md


# Apple Inc. (AAPL) - Agent Research Report

*Generated 2026-10-09T10:58:44+00:00 | session `7805832401ec` | status: **validated** | report revisions: 0 | tools used: calculate_volatility, get_news, get_price_data, llm_sentiment*

*Agent turns by model: openai/gpt-oss-120b x6 | report written by: openai/gpt-oss-120b*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## 1. Financial Health Summary

Apple’s stock is trading at $340.42, just 1.4% below its 52‑week high of $345.34, with a 40% rally from the 52‑week low and a strong YTD return of +25.6% (get_price_data). Momentum indicators show price above the 50‑day SMA ($322.29) and 200‑day SMA ($290.07), RSI at 61.2 (neutral) and MACD histogram slightly negative, while the price sits near the upper Bollinger Band (BB %B 0.89) suggesting limited upside (get_price_data). Valuation remains premium with a trailing P/E of 39.0× and forward P/E of 35.5×, reflecting about 10% earnings growth priced in (get_price_data). Profitability is high with a 27.6% profit margin, earnings growth of 28.7% YoY and ROE of 148.8%, supported by robust free cash flow of $107.7 bn (get_price_data). Balance sheet shows debt/equity of 78.4% and a current ratio of 1.00, indicating moderate leverage but sufficient liquidity (get_price_data).

**Market sentiment.** The LLM sentiment model assigns a neutral score of 0.1508, reflecting mixed news tone. Positive signals come from purchases by Wedmont Private Capital and Endeavor Private Wealth, while a negative headline flags slowing iPhone 18 Pro demand (get_news).

| Metric | Value | Source tool |
|---|---|---|
| Current price | $340.42 | `get_price_data` |
| Trailing P/E | 39.0× | `get_price_data` |
| Free cash flow | $107.7 bn | `get_price_data` |

## 2. Top Three Risks (next 90 days)

### Risk 1: Valuation premium

The stock trades at a high trailing P/E of 39.0×, well above its 5‑year average, leaving limited upside if earnings growth slows.

Evidence:
- `get_price_data`: Trailing P/E = 39.0× (above 5‑yr avg ~28×)

### Risk 2: Demand slowdown

Analyst headlines suggest iPhone 18 Pro demand may be weakening, with Apple cutting component orders, which could pressure revenue.

Evidence:
- `get_news`: Headline: "Is iPhone 18 Pro Demand Slowing? Apple Reportedly Cuts October Component Orders, Stock Slips Premarket"

### Risk 3: High volatility

Annualised volatility of 26.36% implies a potential 90‑day price swing of about 13.1%, increasing risk of sharp moves.

Evidence:
- `calculate_volatility`: Annualised volatility = 26.36%; expected 90‑day move = 13.08%

## 3. Hedge Strategy Recommendation

**Strategy:** Protective put  
**Instrument:** AAPL 90‑day put, strike near 1‑sigma low $298.69  
**Volatility basis:** annualised 26.36%, 90-day 1-sigma move 13.08%

**Rationale.** Mitigates downside from valuation premium and demand slowdown while matching the 13% expected 90‑day move.

**Sizing.** Buy 1 put contract (covering 100 shares) at $298.7 strike, representing ~12% of current price

**Trade-offs.** Cost of premium reduces upside; if stock rises to near 52‑wk high the put may expire worthless.

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


### 3A evidence summary

The table below is computed from the two runs above, not typed by hand.

In [15]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from langchain_core.messages import AIMessage, ToolMessage

from task3_agentic.src.agent import count_tool_rounds


def cycles(messages):
    """Decisions taken right after observing at least one tool result."""
    # zip(messages, messages[1:]) pairs every message with the one after it; a tool result followed by
    # an agent turn is one observe -> decide cycle.
    return sum(1 for prev, cur in zip(messages, messages[1:]) if isinstance(prev, ToolMessage) and isinstance(cur, AIMessage))


def errored_tools(session):
    """Tools that returned error or empty at least once in a session."""
    return sorted({r.tool for r in session.history if r.result.status != "ok"})


report = run.final.report
summary_rows = [
    ("Tools implemented and called", f"{len(set(run.tool_sequence))} distinct of 5 in the main run; all 5 called directly in section 2"),
    ("Order chosen by the model", " -> ".join(run.tool_sequence)),
    ("Order under faults", " -> ".join(fault_run.tool_sequence)),
    ("Observe -> decide cycles", f"main run {cycles(run.messages)}, fault run {cycles(fault_run.messages)}"),
    ("Report-check replans", f"main run {run.final.revisions}, fault run {fault_run.final.revisions}"),
    ("Report status", f"main {run.final.status}, fault {fault_run.final.status}"),
    ("Report sections", "Financial Health Summary / Top Three Risks / Hedge Strategy" if report else "missing"),
    ("Evidence items per risk", [len(r.evidence) for r in report.risks] if report else "-"),
    ("Tools that errored in the fault run", errored_tools(fault_ctx)),
    ("Tools that never recovered (recorded in data_gaps)", sorted(fault_ctx.failed_tools())),
    ("Models that answered", f"main {run.final.agent_models} / report {run.final.report_model}; fault {fault_run.final.agent_models} / report {fault_run.final.report_model}"),
    ("Unhandled exceptions", "none (both runs returned a FinalReport)"),
    ("Trace lines written", len(TraceLogger.read(config.TRACE_FILE))),
]
with pd.option_context("display.max_colwidth", 200):
    display(pd.DataFrame(summary_rows, columns=["Check", "Result"]))

,Check,Result
0,Tools implemented and called,5 distinct of 5 in the main run; all 5 called directly in section 2
1,Order chosen by the model,get_price_data -> get_news -> llm_sentiment -> calculate_volatility -> web_search
2,Order under faults,get_price_data -> get_price_data -> calculate_volatility -> get_news -> llm_sentiment
3,Observe -> decide cycles,"main run 5, fault run 5"
4,Report-check replans,"main run 0, fault run 0"
5,Report status,"main validated, fault validated"
6,Report sections,Financial Health Summary / Top Three Risks / Hedge Strategy
7,Evidence items per risk,"[2, 3, 1]"
8,Tools that errored in the fault run,[get_price_data]
9,Tools that never recovered (recorded in data_gaps),[]


# Part 3B - Multi-Agent Coordination

Two agents with separate roles, separate conversations and separate tool access:

| Agent | Role | Tools (enforced) | Output |
|---|---|---|---|
| A, `data_analyst` | Quantitative analysis | `get_price_data`, `calculate_volatility`, `llm_sentiment` | `DataBrief` (structured JSON) |
| B, `research_writer` | Qualitative synthesis | `web_search`, `get_news` | Final `ResearchReport` |

```
orchestrator --task--> A --DataBrief--> B --ClarificationRequest--> A --ClarificationResponse--> B --ResearchReport--> done
```

**How Agent A gets headlines.** A has `llm_sentiment` but no news access, so its first brief carries no sentiment. B fetches the headlines. B's one clarification request carries them back to A, A scores them, and B uses the score. Neither agent can produce the sentiment evidence alone.

## 6. Handoff Schemas

Every inter-agent message is a Pydantic model inside an `AgentMessage` envelope (sender, recipient, kind, payload, timestamp). Nothing is passed as a raw string. The receiver re-validates the payload against the schema it expects (`AgentMessage.open`).

The **numbers** in `DataBrief` and `ClarificationResponse` are filled in code from Agent A's own tool results. A's LLM writes only the interpretive fields (`key_findings`, `quant_risk_flags`, `confidence_notes`, `answer`).

In [16]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import schema_table
from task3_agentic.src.handoff import AgentMessage, ClarificationRequest, ClarificationResponse, DataBrief

for schema in (DataBrief, ClarificationRequest, ClarificationResponse, AgentMessage):
    print(f"\n{schema.__name__}")
    with pd.option_context("display.max_colwidth", 110):
        display(schema_table(schema))


DataBrief


,field,type,required,description
0,ticker,str,True,
1,as_of,str,True,UTC time the brief was assembled.
2,prepared_by,str,False,
3,price,Optional,False,[code] get_price_data result without raw bars.
4,volatility,Optional,False,[code] calculate_volatility result.
5,sentiment,Optional,False,[code] llm_sentiment result; empty until B sends headlines.
6,sentiment_unavailable_reason,Optional,False,[code] why there is no sentiment yet.
7,tool_status,Dict,False,[code] per A tool: ok / empty / error / not_called.
8,data_gaps,List,False,[code] failed tools and missing blocks.
9,key_findings,List,False,[A's LLM] findings combining several metrics.



ClarificationRequest


,field,type,required,description
0,request_kind,Literal,True,score_headlines: A scores B's headlines with llm_sentiment. volatility_window: A computes volatility over ...
1,question,str,True,The specific question for Agent A.
2,why_needed,str,True,Which part of the report depends on the answer.
3,headlines,List,False,score_headlines only: the headline titles to score.
4,window,Optional,False,volatility_window only: look-back in trading days.
5,period,Optional,False,price_period only: one of the yfinance periods.
6,metric,Optional,False,metric_check only: the brief field to confirm.



ClarificationResponse


,field,type,required,description
0,request_kind,Literal,True,
1,status,Literal,True,[code] ok when A's new tool results hold the requested data.
2,answer,str,True,[A's LLM] 1-3 sentence reply citing the values.
3,sentiment,Optional,False,[code] score_headlines: llm_sentiment on B's headlines.
4,volatility,Optional,False,[code] volatility_window: the requested window.
5,period_return,Optional,False,[code] price_period: the requested period's return.
6,metric_name,Optional,False,[code] metric_check: the metric looked up.
7,metric_value,Optional,False,[code] metric_check: its value.
8,tools_used,List,False,[code] A's tool calls after the request.
9,fulfilled_by,Literal,False,"agent, or fallback if A's own toolkit was run for it."



AgentMessage


,field,type,required,description
0,id,str,True,
1,sender,str,True,"data_analyst, research_writer or orchestrator."
2,recipient,str,True,
3,kind,Literal,True,task / data_brief / clarification_request / clarification_response / final_report.
4,summary,str,True,One line for the trace.
5,payload,Dict,True,"The validated model, dumped; the receiver validates it again."
6,timestamp,str,True,
7,fallback,bool,False,True when the orchestrator's fallback produced this message.


**Tool restriction is enforced in code, three ways:**

1. **Binding.** Each agent's LLM is bound only to its own tool schemas, so it never sees the others.
2. **ToolNode.** Each agent's loop can only execute its own tools. A call to any other name, which a confused or manipulated model might forge, comes back as an error message, and the tool never runs.
3. **Toolkit guard.** Each agent's `ResearchTools` has an `allowed` set, checked in `run_traced`. Even code that calls the toolkit directly is refused, and the attempt is written to `agent_trace.jsonl`.

The cell below exercises all three without any LLM call. The forged call in layer 2 comes from a two-line scripted model.

A related guard: `llm_sentiment` only scores headlines that `get_news` or `web_search` actually returned in this session. This was added after a live run in which Agent A, which has no news access, invented headlines and scored them.

In [17]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableLambda

from task3_agentic.src.multi_agent import build_agent_toolkits
from task3_agentic.src.react import build_react_loop, run_react_loop

A_NAME, B_NAME = config.AGENT_A_NAME, config.AGENT_B_NAME
demo_ctx = SessionContext(config.TICKER, trace=trace)
demo_toolkits = build_agent_toolkits(demo_ctx)

# Layer 1: the tool schemas each agent's LLM is bound to.
display(pd.DataFrame(
    [{"agent": agent, "bound tools": ", ".join(t.name for t in tools)} for agent, (_, tools) in demo_toolkits.items()]
))

# Layer 2: a model forges a call to a tool its agent does not have; the ToolNode refuses it.
# A scripted stand-in for the LLM: each call returns the next message from this list. The first turn
# requests web_search (not one of Agent A's tools); the second ends the loop.
forged = iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "Apple"}, "id": "forged_1", "type": "tool_call"}]),
    AIMessage(content="done"),
])
loop_a = build_react_loop(RunnableLambda(lambda _: next(forged)), demo_toolkits[A_NAME][1], "Agent A", max_rounds=2)
# Run the loop and take the first tool reply: the ToolNode's refusal of the forged call.
rejected = next(m for m in run_react_loop(loop_a, [HumanMessage(content="go")])["messages"] if isinstance(m, ToolMessage))
print(f"Layer 2  data_analyst -> web_search via ToolNode: status={rejected.status}: {rejected.content[:120]}")

# Layer 3: direct calls past the LLM are refused by the toolkit guard (and logged).
toolkit_a, toolkit_b = demo_toolkits[A_NAME][0], demo_toolkits[B_NAME][0]
for agent, result in (
    (A_NAME, toolkit_a.web_search("Apple analyst outlook")),
    (A_NAME, toolkit_a.get_news(config.TICKER, 5)),
    (B_NAME, toolkit_b.get_price_data(config.TICKER, "1y")),
):
    print(f"Layer 3  {agent} -> {result.tool}: status={result.status}: {result.error}")

# Headline guard: A cannot score a headline no news tool returned.
invented = toolkit_a.llm_sentiment(["Apple reports Q3 earnings beat expectations"])
print(f"Guard    data_analyst -> llm_sentiment(invented headline): status={invented.status}: {invented.error}")

denied = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=demo_ctx.session_id)
with pd.option_context("display.max_colwidth", 80):
    display(denied[["agent", "tool", "status", "output"]])

,agent,bound tools
0,data_analyst,"get_price_data, calculate_volatility, llm_sent..."
1,research_writer,"web_search, get_news"


Layer 2  data_analyst -> web_search via ToolNode: status=error: Error: web_search is not a valid tool, try one of [get_price_data, calculate_volatility, llm_sentiment].
10:58:45 | WARNING | task3.tracing | Blocked: data_analyst may not call web_search (allowed: calculate_volatility, get_price_data, llm_sentiment)
10:58:45 | WARNING | task3.tracing | Blocked: data_analyst may not call get_news (allowed: calculate_volatility, get_price_data, llm_sentiment)
10:58:45 | WARNING | task3.tracing | Blocked: research_writer may not call get_price_data (allowed: get_news, web_search)
Layer 3  data_analyst -> web_search: status=error: ToolAccessError: data_analyst may not call web_search (allowed: calculate_volatility, get_price_data, llm_sentiment)
Layer 3  data_analyst -> get_news: status=error: ToolAccessError: data_analyst may not call get_news (allowed: calculate_volatility, get_price_data, llm_sentiment)
Layer 3  research_writer -> get_price_data: status=error: ToolAccessError: research_wri

,agent,tool,status,output
0,data_analyst,web_search,error,"{""status"":""error"",""tool"":""web_search"",""error"":""ToolAccessError: data_analyst..."
1,data_analyst,get_news,error,"{""status"":""error"",""tool"":""get_news"",""error"":""ToolAccessError: data_analyst m..."
2,research_writer,get_price_data,error,"{""status"":""error"",""tool"":""get_price_data"",""error"":""ToolAccessError: research..."
3,data_analyst,llm_sentiment,error,"{""status"":""error"",""tool"":""llm_sentiment"",""error"":""None of the headlines came..."


## 7. Agent A (Data Analyst) and Agent B (Research Writer)

Each agent is a ReAct loop (`src/react.py`, shared with 3A): the LLM picks its own tool calls from its own tool list. The parent graph below runs the four steps in order. Within each step the agent decides what to call.

| Step | Agent | Tool rounds | Structured output |
|---|---|---|---|
| `analyst_initial` | A | up to 5 | `AnalystInterpretation`, merged into the code-filled `DataBrief` |
| `writer_research` | B | up to 5 | exactly one `ClarificationRequest` (B's LLM picks kind and question) |
| `analyst_clarify` | A, same conversation | up to 3 more | `ClarificationResponse` (values from A's new tool results) |
| `writer_final` | B | none | `ResearchReport`, checked by `check_report` + `check_incorporation`, one repair pass |

In [18]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.multi_agent import build_pipeline_graph, default_pipeline_models

preview_ctx = SessionContext(config.TICKER, trace=TraceLogger(None))
preview = build_agent_toolkits(preview_ctx)
# Built only to draw the graph: a throwaway session with an in-memory trace, and observer=None (the last
# argument), so nothing is printed and no LLM is called.
pipeline = build_pipeline_graph(
    preview_ctx,
    default_pipeline_models(preview[A_NAME][1], preview[B_NAME][1]),
    preview[A_NAME][0],
    preview[A_NAME][1],
    preview[B_NAME][1],
    config.RESEARCH_QUERY_TEMPLATE.format(ticker=config.TICKER),
    None,
)
print(pipeline.get_graph().draw_mermaid())
for title, text in (("Agent A system prompt", prompts.ANALYST_SYSTEM), ("Agent B system prompt", prompts.WRITER_SYSTEM)):
    print(f"--- {title} ---")
    print(text.format(today="<run date>", ticker=config.TICKER))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	analyst_initial(analyst_initial)
	writer_research(writer_research)
	analyst_clarify(analyst_clarify)
	writer_final(writer_final)
	__end__([<p>__end__</p>]):::last
	__start__ --> analyst_initial;
	analyst_clarify --> writer_final;
	analyst_initial --> writer_research;
	writer_research --> analyst_clarify;
	writer_final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

--- Agent A system prompt ---
You are Agent A, the Data Analyst in a two-agent equity research team. You do quantitative analysis only.

Your tools: get_price_data(ticker, period), calculate_volatility(ticker, window), llm_sentiment(headlines).
You have no news or web access. Headlines are sent to you by Agent B, the Research Writer. Do not call llm_sentiment until Agent B has sent you headlines, and never write headlines yourself: the tool rejects any headli

**Reading the graph.** The 3B pipeline runs four steps in a fixed order: Agent A's analysis, Agent B's research, A answering B's clarification request, then B's final report. Only the handoff order is fixed; inside each step, the agent chooses its own tools.

## 8. End-to-End Pipeline Run with Full Message Trace

One call runs the whole pipeline: query in, report out, with no manual step. The trace prints live:

- `[A data_analyst]` and `[B research_writer]` prefix each agent's DECIDE / CALL / OBSERVE lines.
- A `HANDOFF` block shows every inter-agent message: sender, recipient, kind, and the validated payload.
- `[pipeline]` lines are graph events (budget reached, model fallbacks, repair passes).

In [19]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import MultiAgentPrinter
from task3_agentic.src.multi_agent import run_multi_agent

from task3_agentic.src.cache import multi_agent_with_cache

ma_ctx = SessionContext(config.TICKER, trace=trace)
# force_refresh: this run is always live (a same-day re-execution must not load yesterday's work from the
# cache); it also writes the cache that section 11 reads. AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)
# One call runs the whole two-agent pipeline (no manual steps). MultiAgentPrinter prints each agent's
# messages and every handoff live; the finished run is in first_cache.run.
first_cache = multi_agent_with_cache(ma_ctx, force_refresh=True, events=events, observer=MultiAgentPrinter())
ma_run = first_cache.run
print("\nCache:", first_cache.note)
print("\nStatus:", ma_run.final.status, "| report repair passes:", ma_run.final.revisions, "| duration:", ma_run.duration_s, "s")
print("Agent turns by model:", ma_run.final.agent_models, "| report written by:", ma_run.final.report_model)
print("Graph events:", ma_run.events or "none")

HANDOFF #1  orchestrator -> data_analyst  [task]
  summary: Research AAPL: quantitative part
  payload: {"ticker": "AAPL", "query": "Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.", "focus": "Quantitative evidence: trend, indicators, fundamentals, and volatility for sizing a 90-day hedge.", "tools": ["get_price_data", "calculate_volatility", "llm_sentiment"]}
[A data_analyst]    DECIDE   [step 1 | openai/gpt-oss-120b]
[A data_analyst]             (reasoning) We need price data then volatility. No headlines yet, so skip sentiment. Call get_price_data.
[A data_analyst]    CALL     get_price_data({"period": "1y", "ticker": "AAPL"})
[A data_analyst]    OBSERVE  get_price_data -> Apple Inc.: price 340.42, 1y return 32.4%, RSI 61.23, momentum Strong Bullish, fwd P/E 35.506, margin 27.62% [source=live]
[A data_analyst]    DECIDE   [step 2 | openai/gpt-oss-120b]

In [20]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import handoff_table

print("Inter-agent messages (in order):")
with pd.option_context("display.max_colwidth", 120):
    display(handoff_table(ma_run.handoffs))
print("Tool calls, with the agent that made each one:")
display(tool_call_table(ma_ctx))
lines = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=ma_ctx.session_id)
# Group this session's trace lines by agent and list each agent's tools in order: {agent: [tools]}.
print("agent_trace.jsonl lines for this run, per agent:", lines.groupby("agent")["tool"].apply(list).to_dict())

Inter-agent messages (in order):


,#,sender,recipient,kind,fallback,payload fields,summary
0,1,orchestrator,data_analyst,task,False,"ticker, query, focus, tools",Research AAPL: quantitative part
1,2,data_analyst,research_writer,data_brief,False,"ticker, as_of, prepared_by, price, volatility, sentiment_unavailable_reason, tool_status,","price 340.42, 1y return 32.4%; 60d vol 26.36% (normal); sentiment not available (no news access); 4 findings, 5 risk..."
2,3,research_writer,data_analyst,clarification_request,False,"request_kind, question, why_needed, headlines",score_headlines: Please score the sentiment of the following headlines using the llm_sentiment tool.
3,4,data_analyst,research_writer,clarification_response,False,"request_kind, status, answer, sentiment, tools_used, fulfilled_by",ok: sentiment 0.1505 (Neutral) from 10 headlines (fulfilled by agent)
4,5,research_writer,orchestrator,final_report,False,"ticker, financial_health, risks, hedge, data_gaps, clarification_used",report on AAPL: Valuation compression; Technical overbought pullback; Volatility‑driven downside


Tool calls, with the agent that made each one:


,#,agent,tool,args,status,source,duration_ms
0,1,data_analyst,get_price_data,"{""ticker"": ""AAPL"", ""period"": ""1y""}",ok,live,511
1,2,data_analyst,calculate_volatility,"{""ticker"": ""AAPL"", ""window"": 60}",ok,session,3
2,3,research_writer,get_news,"{""ticker"": ""AAPL"", ""n"": 15}",ok,live,946
3,4,research_writer,web_search,"{""query"": ""Apple analyst price target 2026 Q4 ...",ok,ddgs,1883
4,5,data_analyst,llm_sentiment,"{""headlines"": [""Why Are Nasdaq Futures Surging...",ok,groq/openai/gpt-oss-120b,31845


agent_trace.jsonl lines for this run, per agent: {'data_analyst': ['get_price_data', 'calculate_volatility', 'llm_sentiment'], 'research_writer': ['get_news', 'web_search']}


In [21]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.multi_agent import save_handoffs

company = ma_ctx.company_names.get(config.TICKER)
ma_md, _ = save_report(ma_run.final, company, suffix=config.MULTI_AGENT_REPORT_SUFFIX)
handoffs_path = save_handoffs(ma_run)
print("Saved:", ma_md.relative_to(config.REPO_ROOT), "and", handoffs_path.relative_to(config.REPO_ROOT))
display(Markdown(render_markdown(ma_run.final, company)))

Saved: task3_agentic/outputs/AAPL_2026-10-09_research_report_multi_agent.md and task3_agentic/outputs/AAPL_2026-10-09_handoffs.json


# Apple Inc. (AAPL) - Agent Research Report

*Generated 2026-10-09T11:03:28+00:00 | session `4de1279b4725` | status: **validated** | report revisions: 0 | tools used: calculate_volatility, get_news, get_price_data, llm_sentiment, web_search*

*Agent turns by model: openai/gpt-oss-120b x8 | report written by: openai/gpt-oss-120b*

*Pipeline: data_analyst: calculate_volatility, get_price_data, llm_sentiment | research_writer: get_news, web_search*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## 1. Financial Health Summary

Apple is trading at $340.42, just 1.42% below its 52‑week high of $345.34, with a strong bullish momentum label (score 0.71) and price above both the 50‑day SMA ($322.29) and 200‑day SMA ($290.07). Valuation remains premium with a forward P/E of 35.506 versus a trailing P/E of 39.039, while profit margin is 27.62% and ROE is an exceptional 148.75%. The balance sheet shows a debt‑to‑equity of 78.445% and a current ratio of 1.003, indicating balanced leverage but thin liquidity. Technical indicators show RSI at 61.23 (neutral‑to‑overbought) and MACD histogram negative, suggesting possible short‑term pullback. Overall, the stock combines high growth profitability with elevated volatility (annualised 26.36%).

**Market sentiment.** The llm_sentiment analysis of ten recent headlines gave a modestly positive score of +0.1505 (Neutral), reflecting mixed but slightly upbeat coverage. Positive headlines such as “Apple gains share even as PC market slumps in Q3” and “Apple (AAPL) Has a New Catalyst That Could Drive Its Next Upgrade Cycle” support the bullish trend, while negative pieces like “Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run” and “Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates” highlight valuation concerns.

| Metric | Value | Source tool |
|---|---|---|
| Current Price | $340.42 | `get_price_data` |
| Forward P/E | 35.506 | `get_price_data` |
| Annualised Volatility (60d) | 26.36% | `calculate_volatility` |

## 2. Top Three Risks (next 90 days)

### Risk 1: Valuation compression

High forward P/E of 35.5 and headlines suggesting the stock is fully priced could trigger a pull‑back if earnings guidance weakens.

Evidence:
- `get_price_data`: Forward P/E = 35.506 (high for a mature consumer‑tech giant)
- `get_news`: Headline: "Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run" (negative sentiment)

### Risk 2: Technical overbought pullback

RSI at 61.23 and price near the upper Bollinger band (BB %B 0.89) raise the risk of a short‑term correction.

Evidence:
- `get_price_data`: RSI_14 = 61.23 (neutral‑to‑overbought)
- `get_price_data`: Bollinger %B = 0.89 (price near upper band)
- `get_news`: Headline: "Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates" (negative)

### Risk 3: Volatility‑driven downside

The 60‑day annualised volatility of 26.36% implies a 90‑day 1‑sigma move of about 13.08%, matching the recent 1‑year max drawdown of –13.8%, exposing the share to sizable swings.

Evidence:
- `calculate_volatility`: Annualised volatility = 26.36% (60‑day)
- `calculate_volatility`: Expected 90‑day move = 13.08% (1‑sigma)
- `get_price_data`: Max drawdown 1y = -13.8% (matches expected move)

## 3. Hedge Strategy Recommendation

**Strategy:** Protective put  
**Instrument:** AAPL 90-day put, strike near the 1‑sigma low ($298.70)  
**Volatility basis:** annualised 26.36%, 90-day 1-sigma move 13.08%

**Rationale.** Protects against the downside risk implied by the 13.08% 90‑day move and valuation‑compression scenario while allowing upside if price stays above the strike.

**Sizing.** Buy 1 put contract (100 shares) with strike $298.70, covering ~12% of the current price

**Trade-offs.** Cost of the put premium reduces net upside; if the stock rallies above $340 the hedge may expire worthless.

## Clarification from the Data Analyst

Requested sentiment scoring of ten headlines (request_kind: score_headlines). Agent A returned a confidence‑weighted sentiment score of +0.1505 (label Neutral) with top positive and negative headlines. These scores are used in the market_sentiment paragraph and to support the valuation‑compression risk.

## Data gaps

- llm_sentiment tool was not called in the original data brief; sentiment was obtained via a clarification request.

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


## 9. Critique Loop: Request, Response, Incorporation

The critique loop from the run above: B's one request, A's answer, and where B's final report uses it. `check_incorporation` reads every number in the report and confirms that A's returned value is among them. It also requires `clarification_used` to say what was asked and how the answer was used.

In [22]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.handoff import check_incorporation, expected_value

# The first (and only) handoff of each kind from the run.
request_message = next(h for h in ma_run.handoffs if h.kind == "clarification_request")
response_message = next(h for h in ma_run.handoffs if h.kind == "clarification_response")
print(f"REQUEST  {request_message.sender} -> {request_message.recipient}{' (fallback)' if request_message.fallback else ''}")
print(json.dumps(request_message.payload, indent=2, ensure_ascii=False))
answer = dict(response_message.payload)
for key in ("sentiment", "volatility"):
    if key in answer:  # keep the printout short: the full block is in the handoffs file
        answer[key] = {k: answer[key][k] for k in list(answer[key])[:8]}
print(f"\nRESPONSE {response_message.sender} -> {response_message.recipient}")
print(json.dumps(answer, indent=2, ensure_ascii=False))

if ma_run.final.report is not None:
    # expected_value gives (value, tolerance, label): the number from A's answer that B's report must use.
    target = expected_value(ma_run.response)
    print("\nValue B had to use:", target[2] + " = " + str(target[0]) if target else "none (A's answer failed)")
    print("B's clarification_used:", ma_run.final.report.clarification_used)
    print("check_incorporation issues:", check_incorporation(ma_run.final.report, ma_run.response) or "none")

REQUEST  research_writer -> data_analyst
{
  "request_kind": "score_headlines",
  "question": "Please score the sentiment of the following headlines using the llm_sentiment tool.",
  "why_needed": "Sentiment scores are needed to support the qualitative analysis and risk assessment sections of the report.",
  "headlines": [
    "Why Are Nasdaq Futures Surging Premarket? SPCX, ASTS, AAPL, TMUS, VZ, NVDA, MU, LITE Stocks In Focus",
    "Apple Inc. $AAPL Stock Sold by Tanager Wealth Management LLP",
    "Apple Inc. $AAPL Stock Bought by Endeavor Private Wealth Inc.",
    "Apple Inc. $AAPL Stock Purchased by Wedmont Private Capital",
    "Apple (AAPL) legal chief sells shares at $332.01 under a prearranged trading plan.",
    "Apple gains share even as PC market slumps in Q3, IDC says (AAPL:NASDAQ)",
    "Apple Rises 24% YTD: Should You Buy, Sell or Hold the AAPL Stock?",
    "Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates",
    "Apple (AAPL) Has a New Catalyst That Could

### 3B evidence summary

Computed from the run above and the section 6 enforcement demo.

In [23]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.handoff import AnalystTask
from task3_agentic.src.schemas import ResearchReport

expected_schema = {
    "task": AnalystTask,
    "data_brief": DataBrief,
    "clarification_request": ClarificationRequest,
    "clarification_response": ClarificationResponse,
    "final_report": ResearchReport,
}
# Re-parse every handoff payload with the schema for its kind. model_validate raises on any mismatch, so
# reaching True means every message was valid structured data, not free text.
revalidated = all(expected_schema[h.kind].model_validate(h.payload) is not None for h in ma_run.handoffs)
# Any tool call an agent made outside its own tool list (expected: none).
crossed = [r.tool for r in ma_ctx.history if r.tool not in (config.AGENT_A_TOOLS if r.agent == A_NAME else config.AGENT_B_TOOLS)]
demo_denied = [r for r in demo_ctx.history if r.denied]
rows = [
    ("Bound tools", ma_run.bound_tools),
    ("Tool calls outside an agent's list in the run", crossed or "none"),
    ("Calls refused by the guard in the section 6 demo", len(demo_denied)),
    ("Handoff kinds, in order", " -> ".join(h.kind for h in ma_run.handoffs)),
    ("Every payload re-validates against its schema", revalidated),
    ("Critique rounds", ma_run.critique_rounds),
    ("Request", f"{ma_run.request.request_kind}: {ma_run.request.question}" if ma_run.request else "-"),
    ("Answer", f"{ma_run.response.status}, fulfilled by {ma_run.response.fulfilled_by}" if ma_run.response else "-"),
    ("Incorporation issues", check_incorporation(ma_run.final.report, ma_run.response) if ma_run.final.report else "no report"),
    ("Tool calls per agent", {a: ma_run.tool_sequence(a) for a in (A_NAME, B_NAME)}),
    ("Final status", ma_run.final.status),
    ("Manual steps between query and report", 0),
]
with pd.option_context("display.max_colwidth", 200):
    display(pd.DataFrame(rows, columns=["Check", "Result"]))

,Check,Result
0,Bound tools,"{'data_analyst': ['get_price_data', 'calculate_volatility', 'llm_sentiment'], 'research_writer': ['web_search', 'get_news']}"
1,Tool calls outside an agent's list in the run,none
2,Calls refused by the guard in the section 6 demo,3
3,"Handoff kinds, in order",task -> data_brief -> clarification_request -> clarification_response -> final_report
4,Every payload re-validates against its schema,True
5,Critique rounds,1
6,Request,score_headlines: Please score the sentiment of the following headlines using the llm_sentiment tool.
7,Answer,"ok, fulfilled by agent"
8,Incorporation issues,[]
9,Tool calls per agent,"{'data_analyst': ['get_price_data', 'calculate_volatility', 'llm_sentiment'], 'research_writer': ['get_news', 'web_search']}"


# Part 3C - Memory and Observability

| Requirement | Mechanism | Shown in |
|---|---|---|
| Short-term memory | The 3A research graph is compiled with a LangGraph checkpointer (`InMemorySaver`). The follow-up continues the same thread, so the agent's earlier tool results are in its context. | §10 |
| Persistent cache | `cache/<TICKER>_<YYYY-MM-DD>_<pipeline>.json`, date in US/Eastern. A same-day call loads it with no tool calls. | §8 (write), §11 (hit) |
| `agent_trace.jsonl` | Every tool call: timestamp, session, agent, tool, args, status, output (200 chars), duration. Cache and memory events go to `logs/session_events.jsonl`. | §12 |

## 10. Short-Term Memory: Follow-Up Question Without Re-Calling Tools

The follow-up is a new message on the **section 4 run's own thread**. The agent has its five tools bound and may call one (it gets one tool round, then must answer). It is told that its earlier tool results are its memory. In follow-up mode, an answer without a tool call ends the turn, so no new report is written.

The evidence below is computed, not asserted:
- tool calls during the follow-up;
- `agent_trace.jsonl` line count before and after;
- whether the answer cites the values of the earlier `calculate_volatility` result.

In [24]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.display import followup_table
from task3_agentic.src.memory import ask_followup, volatility_memory

question = "Which volatility window did you compute, and what 90-day 1-sigma price band did it give?"
# The checkpointer stores each conversation under a thread_id. Reading the section 4 thread shows the
# messages the agent still has in memory before the follow-up is asked.
thread = {"configurable": {"thread_id": run.thread_id}}
in_thread = len(run.graph.get_state(thread).values.get("messages", [])) if run.graph is not None else 0
print(f"Thread {run.thread_id}: {in_thread} messages already in memory (the section 4 run)")
remembered = volatility_memory(run)
print("Earlier calculate_volatility result:", remembered or "none (section 4 did not compute volatility)")

# The follow-up continues that same thread. `remembered` holds the earlier volatility values, so the
# table below can check which of them the answer cites.
followup = ask_followup(run, question, remembered=remembered, on_message=TracePrinter(), events=events)
print("\nAnswer:", followup.answer or followup.error)
with pd.option_context("display.max_colwidth", 140):
    display(followup_table(followup))

Thread 3e5a9005bdcd: 12 messages already in memory (the section 4 run)
Earlier calculate_volatility result: {'window': 60.0, 'annualised_vol_pct': 26.36, 'expected_move_pct': 13.08, 'band_1sigma_low': 298.69, 'band_1sigma_high': 387.98}
----------------------------------------------------------------------------------------------------
QUERY    [Follow-up question] Which volatility window did you compute, and what 90-day 1-sigma price band did it give? The tool results earlier in this conversation are your memory. Answer from them, cite the exact values, and name the tool result you used. Call a tool only if those results do not contain the answer. Reply in 2-4 sentences.
----------------------------------------------------------------------------------------------------
DECIDE   [step 1 | openai/gpt-oss-120b]
         I calculated volatility using the **`calculate_volatility`** tool with a **60‑day look‑back window**. The output gave a 90‑day 1‑σ expected move of ±13.08 % and a corres

,Check,Result
0,Question,"Which volatility window did you compute, and what 90-day 1-sigma price band did it give?"
1,Answered by,openai/gpt-oss-120b
2,Tool calls during the follow-up,none
3,agent_trace.jsonl lines before -> after,28 -> 28
4,Answered from memory,True
5,Error,none
6,Cites remembered window = 60.0,yes
7,Cites remembered annualised_vol_pct = 26.36,no
8,Cites remembered expected_move_pct = 13.08,yes
9,Cites remembered band_1sigma_low = 298.69,yes


## 11. Persistent Cache: Second Run Loads the Cached Brief

Section 8 ran the two-agent pipeline live (`force_refresh=True`) and wrote today's brief to the cache. The cell below asks for the same ticker again **as a new session**. `multi_agent_with_cache` finds `AAPL_<today>_multi_agent.json`, validates it, and returns it without running a single tool. A cached file from another date is never reused.

In [25]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

import datetime as dt

from task3_agentic.src.cache import cache_path, load_cached, market_date

written = cache_path(config.TICKER, config.PIPELINE_MULTI_AGENT)
print("Market date (US/Eastern):", market_date())
print("Section 8 cache result:", first_cache.note)
if written.exists():
    stored = json.loads(written.read_text(encoding="utf-8"))
    print(f"{written.relative_to(config.REPO_ROOT)}: {written.stat().st_size:,} bytes")
    display(pd.DataFrame(
        [(key, stored[key]) for key in ("schema_version", "ticker", "cache_date", "pipeline", "created_at", "session_id", "status", "tool_calls")],
        columns=["field", "value"],
    ))

Market date (US/Eastern): 2026-10-09
Section 8 cache result: Ran the pipeline and wrote AAPL_2026-10-09_multi_agent.json
task3_agentic/cache/AAPL_2026-10-09_multi_agent.json: 23,512 bytes


,field,value
0,schema_version,1
1,ticker,AAPL
2,cache_date,2026-10-09
3,pipeline,multi_agent
4,created_at,2026-10-09T11:03:28+00:00
5,session_id,4de1279b4725
6,status,validated
7,tool_calls,5


In [26]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.cache import multi_agent_with_cache
from task3_agentic.src.display import cache_table

if first_cache.brief is None:
    print("Section 8 produced no report, so nothing was cached; skipping the second run rather than spending quota.")
else:
    # A new session asks for the same ticker on the same date. Counting the trace lines before and after
    # proves the cache hit ran no tool.
    second_ctx = SessionContext(config.TICKER, trace=trace)
    lines_before = trace_line_count(config.TRACE_FILE)
    second = multi_agent_with_cache(second_ctx, events=events, observer=MultiAgentPrinter())
    lines_after = trace_line_count(config.TRACE_FILE)
    display(cache_table(second))
    print(f"agent_trace.jsonl lines: {lines_before} -> {lines_after} (no tool ran)" if lines_after == lines_before
          else f"agent_trace.jsonl lines: {lines_before} -> {lines_after}")
    display(Markdown(render_markdown(second.brief.final, ma_ctx.company_names.get(config.TICKER))))

,Item,Value
0,Result,HIT
1,Cache file,AAPL_2026-10-09_multi_agent.json
2,Tool calls made by this call,0
3,Seconds,0.002
4,Note,Loaded AAPL_2026-10-09_multi_agent.json (writt...
5,Cached by session,4de1279b4725
6,Cache date (US/Eastern),2026-10-09
7,Report status,validated
8,Tool calls of the original run,5


agent_trace.jsonl lines: 28 -> 28 (no tool ran)


# Apple Inc. (AAPL) - Agent Research Report

*Generated 2026-10-09T11:03:28+00:00 | session `4de1279b4725` | status: **validated** | report revisions: 0 | tools used: calculate_volatility, get_news, get_price_data, llm_sentiment, web_search*

*Agent turns by model: openai/gpt-oss-120b x8 | report written by: openai/gpt-oss-120b*

*Pipeline: data_analyst: calculate_volatility, get_price_data, llm_sentiment | research_writer: get_news, web_search*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## 1. Financial Health Summary

Apple is trading at $340.42, just 1.42% below its 52‑week high of $345.34, with a strong bullish momentum label (score 0.71) and price above both the 50‑day SMA ($322.29) and 200‑day SMA ($290.07). Valuation remains premium with a forward P/E of 35.506 versus a trailing P/E of 39.039, while profit margin is 27.62% and ROE is an exceptional 148.75%. The balance sheet shows a debt‑to‑equity of 78.445% and a current ratio of 1.003, indicating balanced leverage but thin liquidity. Technical indicators show RSI at 61.23 (neutral‑to‑overbought) and MACD histogram negative, suggesting possible short‑term pullback. Overall, the stock combines high growth profitability with elevated volatility (annualised 26.36%).

**Market sentiment.** The llm_sentiment analysis of ten recent headlines gave a modestly positive score of +0.1505 (Neutral), reflecting mixed but slightly upbeat coverage. Positive headlines such as “Apple gains share even as PC market slumps in Q3” and “Apple (AAPL) Has a New Catalyst That Could Drive Its Next Upgrade Cycle” support the bullish trend, while negative pieces like “Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run” and “Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates” highlight valuation concerns.

| Metric | Value | Source tool |
|---|---|---|
| Current Price | $340.42 | `get_price_data` |
| Forward P/E | 35.506 | `get_price_data` |
| Annualised Volatility (60d) | 26.36% | `calculate_volatility` |

## 2. Top Three Risks (next 90 days)

### Risk 1: Valuation compression

High forward P/E of 35.5 and headlines suggesting the stock is fully priced could trigger a pull‑back if earnings guidance weakens.

Evidence:
- `get_price_data`: Forward P/E = 35.506 (high for a mature consumer‑tech giant)
- `get_news`: Headline: "Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run" (negative sentiment)

### Risk 2: Technical overbought pullback

RSI at 61.23 and price near the upper Bollinger band (BB %B 0.89) raise the risk of a short‑term correction.

Evidence:
- `get_price_data`: RSI_14 = 61.23 (neutral‑to‑overbought)
- `get_price_data`: Bollinger %B = 0.89 (price near upper band)
- `get_news`: Headline: "Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates" (negative)

### Risk 3: Volatility‑driven downside

The 60‑day annualised volatility of 26.36% implies a 90‑day 1‑sigma move of about 13.08%, matching the recent 1‑year max drawdown of –13.8%, exposing the share to sizable swings.

Evidence:
- `calculate_volatility`: Annualised volatility = 26.36% (60‑day)
- `calculate_volatility`: Expected 90‑day move = 13.08% (1‑sigma)
- `get_price_data`: Max drawdown 1y = -13.8% (matches expected move)

## 3. Hedge Strategy Recommendation

**Strategy:** Protective put  
**Instrument:** AAPL 90-day put, strike near the 1‑sigma low ($298.70)  
**Volatility basis:** annualised 26.36%, 90-day 1-sigma move 13.08%

**Rationale.** Protects against the downside risk implied by the 13.08% 90‑day move and valuation‑compression scenario while allowing upside if price stays above the strike.

**Sizing.** Buy 1 put contract (100 shares) with strike $298.70, covering ~12% of the current price

**Trade-offs.** Cost of the put premium reduces net upside; if the stock rallies above $340 the hedge may expire worthless.

## Clarification from the Data Analyst

Requested sentiment scoring of ten headlines (request_kind: score_headlines). Agent A returned a confidence‑weighted sentiment score of +0.1505 (label Neutral) with top positive and negative headlines. These scores are used in the market_sentiment paragraph and to support the valuation‑compression risk.

## Data gaps

- llm_sentiment tool was not called in the original data brief; sentiment was obtained via a clarification request.

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


In [27]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

# A different date is a different key: yesterday's file (if any) is never served for today.
# market_date() is a "YYYY-MM-DD" string: convert it to a Timestamp, subtract a day, format it back.
yesterday = (pd.Timestamp(market_date()) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
print(f"load_cached({config.TICKER!r}, multi_agent, date={yesterday!r}):", load_cached(config.TICKER, config.PIPELINE_MULTI_AGENT, date=yesterday))
print(f"load_cached({config.TICKER!r}, single_agent, today):", load_cached(config.TICKER, config.PIPELINE_SINGLE_AGENT))

load_cached('AAPL', multi_agent, date='2026-10-08'): None
load_cached('AAPL', single_agent, today): None


## 12. Trace Log Inspection (`agent_trace.jsonl`)

`agent_trace.jsonl` is written by `run_traced`, the one function every tool call passes through, including calls the tool-access guard refused. `audit_trace` checks every line against the specification: required fields, output of 200 characters or less, and a duration of 0 or more.

In [28]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.dashboard_data import load_events, load_trace, per_session
from task3_agentic.src.tracing import audit_trace

records = TraceLogger.read(config.TRACE_FILE)
audit = audit_trace(records)
display(pd.DataFrame(
    [
        ("Lines (tool calls)", audit["lines"]),
        ("Lines missing a required field", len(audit["missing_fields"])),
        (f"Outputs over {config.TRACE_OUTPUT_MAX_CHARS} characters", len(audit["output_over_limit"])),
        ("Missing or negative durations", len(audit["bad_duration"])),
        ("Calls by agent", audit["by_agent"]),
        ("Calls by tool", audit["by_tool"]),
        ("Calls by status", audit["by_status"]),
    ],
    columns=["Check", "Result"],
))
print("Sessions in this notebook run:")
display(per_session(load_trace(config.TRACE_FILE)))
print("Three raw lines:")
for record in records[:3]:
    print(json.dumps(record, ensure_ascii=False))
print("\nsession_events.jsonl:")
with pd.option_context("display.max_colwidth", 120):
    display(load_events(config.EVENTS_FILE))

,Check,Result
0,Lines (tool calls),28
1,Lines missing a required field,0
2,Outputs over 200 characters,0
3,Missing or negative durations,0
4,Calls by agent,"{'tool_check': 9, 'research_agent': 10, 'data_..."
5,Calls by tool,"{'get_price_data': 8, 'get_news': 5, 'calculat..."
6,Calls by status,"{'ok': 19, 'error': 8, 'empty': 1}"


Sessions in this notebook run:


,session_id,started,agents,calls,not_ok,tool_time_s
0,02d29fc91c64,2026-10-09 10:53:35.358000+00:00,tool_check,9,4,13.34
1,3e5a9005bdcd,2026-10-09 10:53:49.929000+00:00,research_agent,5,0,34.21
2,7805832401ec,2026-10-09 10:56:15.890000+00:00,research_agent,5,1,19.29
3,ec35b4fcd5fd,2026-10-09 10:58:45.319000+00:00,"data_analyst, research_writer",4,4,0.00
4,4de1279b4725,2026-10-09 10:59:01.981000+00:00,"data_analyst, research_writer",5,0,35.19


Three raw lines:
{"timestamp": "2026-10-09T10:53:35.358+00:00", "session_id": "02d29fc91c64", "agent": "tool_check", "tool": "get_price_data", "args": {"ticker": "AAPL", "period": "1y"}, "status": "ok", "output": "{\"status\":\"ok\",\"tool\":\"get_price_data\",\"data\":{\"ticker\":\"AAPL\",\"company_name\":\"Apple Inc.\",\"currency\":\"USD\",\"period\":\"1y\",\"as_of_date\":\"2026-10-08\",\"rows_fetched\":753,\"current_price\":340.42,\"period_re", "duration_ms": 1520.9}
{"timestamp": "2026-10-09T10:53:36.701+00:00", "session_id": "02d29fc91c64", "agent": "tool_check", "tool": "get_news", "args": {"ticker": "AAPL", "n": 10}, "status": "ok", "output": "{\"status\":\"ok\",\"tool\":\"get_news\",\"data\":{\"ticker\":\"AAPL\",\"count\":10,\"headlines\":[{\"title\":\"Why Are Nasdaq Futures Surging Premarket? SPCX, ASTS, AAPL, TMUS, VZ, NVDA, MU, LITE Stocks In Focus\",\"publis", "duration_ms": 1342.3}
{"timestamp": "2026-10-09T10:53:36.714+00:00", "session_id": "02d29fc91c64", "agent": "tool_

,timestamp,session_id,event,detail
0,2026-10-09 10:58:45.714000+00:00,4de1279b4725,cache_bypass,"{""path"": ""AAPL_2026-10-09_multi_agent.json"", ""reason"": ""force_refresh""}"
1,2026-10-09 11:03:28.024000+00:00,4de1279b4725,cache_write,"{""path"": ""AAPL_2026-10-09_multi_agent.json"", ""status"": ""validated"", ""tool_calls"": 5}"
2,2026-10-09 11:04:20.073000+00:00,3e5a9005bdcd,followup_answered,"{""question"": ""Which volatility window did you compute, and what 90-day 1-sigma price band did it give?"", ""tool_calls..."
3,2026-10-09 11:04:20.127000+00:00,19b5644df559,cache_hit,"{""path"": ""AAPL_2026-10-09_multi_agent.json"", ""cached_session"": ""4de1279b4725"", ""status"": ""validated""}"


# Bonus - Observability Platform

## 13. Streamlit Trace Dashboard

A Streamlit dashboard reads `agent_trace.jsonl` and `session_events.jsonl` and shows the trace visually:
- filters by session, agent, tool and status;
- KPIs (calls, errors, empty, denied, p50/p95 duration);
- a per-session table;
- a Gantt-style timeline of each session's tool calls;
- every call's arguments and logged output;
- the cache and memory events.

```
streamlit run task3_agentic/dashboard/app.py
```

The data functions behind it (`src/dashboard_data.py`) are tested offline. The cell below shows the dashboard's KPI row and session table for this notebook run.

In [29]:
# AI-ASSISTED: Claude Code (claude-sonnet-5.5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.dashboard_data import summarise

trace_frame = load_trace(config.TRACE_FILE)
display(pd.DataFrame([summarise(trace_frame)]))

,calls,sessions,errors,empty,denied,total_s,p50_ms,p95_ms
0,28,5,5,1,3,102.04,486.2,26474.2


## Citations

See [`CITATIONS.md`](../CITATIONS.md) in the repository root. AI-assisted cells are also cited inline.